# Paper 5, RB-4 - mixed-frequency RoPE: do single heads still rebuild one axis?

**Naming.** W1-W7 = September runs. RB = November rebuttal runs: RB-0 = single-head run (files `w8_*`, canonical =
cuDNN 9.10.2 re-run of 4 Nov; its part F is the first mixed-RoPE check), RB-1/RB-1b = drift checks, RB-2 = axis map,
RB-3 = intact / shuffled / null. This is **RB-4**; it writes to `./rb4_output/`.

**Why.** The mock review says the row/column split is "true by construction" in *axial* 2D RoPE, where every
rotation plane belongs to one axis, and asks whether the claims hold for *mixed* 2D RoPE (Heo et al., 2024), where
every angle is m_x·f_x + m_y·f_y with learned, per-head, per-block frequencies. RB-0 part F showed on the
recipe-matched mixed ViT-B (`vit_base_patch16_rope_mixed_224.naver_in1k`) that removing one coordinate from every
angle still removes only that axis. RB-4 asks the next question, the one the paper's §4.3 and §4.5 rest on: after a
one-coordinate window, is the axis rebuilt by particular **single heads**, each rebuilding one axis, and do those
heads matter for top-1 only when they have to rebuild? It also asks what axial models cannot: is a head's rebuilt
axis just its learned frequency **orientation**?

| window | blocks 0-5 | reference | axial analogue |
|---|---|---|---|
| `cref` | f_x and f_y zeroed (all rotations identity) | Cref (= RB-0 F) | Cref |
| `row`  | the row coordinate zeroed in every angle | win_row (= RB-0 F) | Dref_row |
| `col`  | the column coordinate zeroed in every angle | win_col (= RB-0 F) | Dref_col |

A single (block, head) in 6-11 is then kept without rotations (both coordinates zeroed for that head in that block):
216 conditions, row/column probes entering block 11 and at the output of block 11 (block 11: output only), exactly
RB-2's protocol. Top-1 (ILSVRC, 5,000) for every single head after the 0-5 window and in the intact model, exactly
RB-0 part C / RB-3 part A. Heads are ranked by RB-2's pre-registered rule (`ssdc_b11`).

**Coherence.** Same env (`env_paper5_cudnn910`), datasets, counts, batch, seed, fp32, readouts and probe protocol as
RB-0/RB-2/RB-3. Edits are made on the learned frequencies (as RB-0 F did) and restored bit for bit. Gates demand
bit-identical agreement with every RB-0 part F number this run repeats. Runtime ~1-1.3 h on one H100.

In [ ]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas"]:
    ensure(p)
print("environment ready")

In [ ]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# config
REPO_PATH  = os.environ.get("RB4_REPO", "")
RB0_DIR    = os.environ.get("RB4_RB0_DIR", "")
RESUME     = os.environ.get("RB4_RESUME", "1") == "1"
STRICT     = os.environ.get("RB4_STRICT", "1") == "1"
TEST_MODE  = os.environ.get("RB4_TEST", "0") == "1"
OUT_DIR    = Path(os.environ.get("RB4_OUT", "rb4_output"))
MODELS_RUN = ["mixed"]

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
ACC_DATASET  = "ILSVRC/imagenet-1k"
N_BLOCKS = 12
WINDOW   = list(range(0, 6))
POST     = list(range(6, 12))
FINAL    = 12
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 / RB-0 / RB-2, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200
P_REF = [7, 8, 9, 10, 11, FINAL]
P_SINGLE = [11, FINAL]
NOISE_AT = (11, FINAL)

EXACT_TOL = 0.0
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_TF32_RANGE = (1e-4, 2e-3)   # RB-1b: TF32 kernels gave 2.7e-4 to 5.1e-4 on the other five models; exact fp32 gives ~1e-6
RB0_SHA256 = {"w8_partF_mixed.json": "82bc61b76766f80c000cf9814a1544e722a128a4aba5871dcb9f13fc5b0c2c48"}
MODEL_SPECS = {"mixed": dict(name="vit_base_patch16_rope_mixed_224.naver_in1k", classifier=True, label="naver ViT-B, mixed RoPE")}
WINDOW_ORDER = ["cref", "row", "col"]
REFN = {"cref": "Cref", "row": "Dref_row", "col": "Dref_col"}
RB0F_NAME = {"baseline": "baseline", "Cref": "Cref", "Dref_row": "win_row", "Dref_col": "win_col"}

# rules, fixed before the run
RULE_BAR = "as RB-2: 3 x the larger combined probe SD at that readout (row/col windows: Dref_row's row probe and Dref_col's column probe; cref: Cref's two probes)"
RULE_LABEL = "as RB-2: row if row cost > bar and row cost - col cost > bar; col likewise; mixed if a cost > bar but no gap; none otherwise"
RULE_RANK = "as RB-2's pre-registered decision: ssdc_b11 for heads in blocks 6-10; block-11 heads separately at the output of block 11"
RULE_ORIENT = ("orientation of (block, head) = (sum_p |f_row| - sum_p |f_col|) / (sum_p |f_row| + sum_p |f_col|) over its 32 "
               "planes, from the learned frequencies; +1 = every plane turns with the row only, -1 = column only, 0 = balanced. "
               "Test: Spearman of orientation with (row cost - col cost) entering block 11 over the 60 heads in blocks 6-10, "
               "permutation p; and whether each specialist's axis matches the sign of its orientation.")
PERM_N, DELTA_BOOT, DELTA_SEED = 10000, 2000, 2026

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    PROBE_PASSES, NOISE_BOOT, PERM_N, DELTA_BOOT = 2, 10, 500, 200
    OUT_DIR = Path(os.environ.get("RB4_OUT", "rb4_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb4_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
print(f"device {DEVICE} | fp32 | model {MODEL_SPECS['mixed']['name']}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

## Pre-flight: are we in `env_paper5_cudnn910`?
A small GPU convolution loads cuDNN; then the notebook checks the version PyTorch reports, which library files were
actually loaded (from the process's own memory map), the installed cuDNN packages, and that the TF32 / cuDNN flags
are PyTorch's defaults (W5 and RB-0 ran with defaults). Nothing is measured if any check fails.

In [ ]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb4_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb4_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

In [ ]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB4_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

## The canonical RB-0 part F (fingerprint check)

In [ ]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

if not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB4_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_pf = (_rb0 / "w8_partF_mixed.json") if _rb0 else None
RB0F = None
if _pf is None or not _pf.exists():
    if TEST_MODE: gate("RB-0 part F present", "mixed", True, "test mode: no RB-0 folder", skip=True)
    else: raise SystemExit(f"RB-0 part F not found: {_pf}")
else:
    if TEST_MODE:
        gate("w8_partF_mixed.json is the canonical RB-0 file", "mixed", True, "test mode: hash not checked", skip=True)
    else:
        d = sha256_file(_pf)
        if not gate("w8_partF_mixed.json is the canonical RB-0 file (cuDNN 9.10.2 re-run)", "mixed", d == RB0_SHA256["w8_partF_mixed.json"],
                    f"sha256 {d[:16]}... (expected {RB0_SHA256['w8_partF_mixed.json'][:16]}...)"):
            raise SystemExit("RB-0 FINGERPRINT FAILED: RB4_RB0_DIR must be the cuDNN 9.10.2 re-run (rb0_cudnn910/w8_output). Nothing was measured.")
    RB0F = json.loads(_pf.read_text())
    gate("RB-0 part F is complete and has the three windows and their top-1", "mixed",
         bool(RB0F.get("complete")) and all(n in RB0F["conditions"] for n in ("baseline", "Cref", "win_row", "win_col"))
         and all(n in RB0F.get("acc", {}) for n in ("baseline", "floor", "Cref", "win_row", "win_col")),
         f"row component = freqs[{RB0F.get('row_component')}]")

In [ ]:
# datasets: explicit ids, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed):
            self.n, self.seed = n, seed
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0, RB-2)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0, RB-3)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)

In [ ]:
# rotary helpers: from W5 / RB-0
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

def win_plus(model, source, heads_by_block, n_heads, window_tf=IDEN, inner=IDEN, window=None):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads. heads_by_block: {block: [heads]},
    blocks outside the window."""
    segs = [(list(WINDOW if window is None else window), window_tf)] if (window is None or window) else []
    for k in sorted(heads_by_block):
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

In [ ]:
# algebra gates (as in W5 / RB-0)
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H_ = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H_, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H_)(e), rh), full, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H_)(e), rh); others = [h for h in range(H_) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head", torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))
    # single head off after a one-axis window (the window masks that axis's planes, the head masks all)
    ew = mask_planes(e, list(range(0, npl, 2)), npl, rh)
    gate(f"one-axis window leaves the other axis's planes bit-identical [{lname}]", "algebra",
         torch.equal(ew.chunk(2, -1)[0][:, plane_columns(1, npl, rh)], e.chunk(2, -1)[0][:, plane_columns(1, npl, rh)]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    _wp = win_plus(_Stub(), "timm", {7: [3]}, 12, window_tf=IDEN)
    dj = [b for b, _ in _wp.segs] == [WINDOW, [7]]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment for the single head's block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

In [ ]:
# mixed-RoPE edits: zero chosen learned frequencies in place, restore exactly
class MixedSpec:
    """Zero coordinate components of the learned mixed-RoPE frequencies freqs[comp, depth, head, plane] in place
    (restored on exit). `masks`: list of (comps, depths, heads), None = all."""
    def __init__(s, rope, masks): s.rope, s.masks = rope, [tuple(m) for m in masks]; s.handles = []
    def __enter__(s):
        f = s.rope.freqs; s.saved = f.detach().clone()
        m = torch.zeros_like(f, dtype=torch.bool)
        for comps, depths, heads in s.masks:
            ds = range(f.shape[1]) if depths is None else depths
            hs = range(f.shape[2]) if heads is None else heads
            for c in comps:
                for d in ds:
                    for h in hs: m[c, d, h, :] = True
        with torch.no_grad(): f[m] = 0.0
        s.handles = [1] * len(s.masks); return s
    def __exit__(s, *e):
        with torch.no_grad(): s.rope.freqs.copy_(s.saved)
        s.handles = []

def window_mask(w, ROWC, COLC):
    return {"cref": ([0, 1], WINDOW, None), "row": ([ROWC], WINDOW, None), "col": ([COLC], WINDOW, None)}[w]

class _FakeRope:   # algebra gates on a stand-in tensor with the real shape convention
    def __init__(s): s.freqs = torch.randn(2, N_BLOCKS, 12, 32)
_fr = _FakeRope(); _orig = _fr.freqs.clone()
with MixedSpec(_fr, [([0, 1], [7], [3])]):
    _chg = (_fr.freqs != _orig)
    ok1 = bool(_chg[:, 7, 3, :].all()) and int(_chg.sum()) == 2 * 32
with MixedSpec(_fr, [([1], WINDOW, None), ([0, 1], [8], [2])]):
    _chg = (_fr.freqs != _orig)
    ok2 = bool(_chg[1, :6].all()) and not bool(_chg[0, :6].any()) and bool(_chg[:, 8, 2].all()) and int(_chg.sum()) == 6 * 12 * 32 + 2 * 32
gate("MixedSpec zeroes exactly one (block, head)'s 2 x 32 frequencies", "algebra", ok1)
gate("MixedSpec composes a one-coordinate window with one (block, head)", "algebra", ok2)
gate("MixedSpec restores the exact tensor", "algebra", torch.equal(_fr.freqs, _orig))

In [ ]:
# image cache: decode once per model, replay through a real DataLoader (RB-0)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_digest(cache):
    h = hashlib.sha1()
    for im, lb in cache[:2]:
        h.update(im[:4].numpy().tobytes()); h.update(lb[:4].numpy().tobytes())
    return h.hexdigest()[:12]

In [ ]:
# measurement: RB-0's measure() without the bootstrap / head-norm extras
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached images under `ctx`: SSDC at every block input and at FINAL, plus captured patch
    tokens."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0])))]
    for i in range(N_BLOCKS):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach()); acc.add(FINAL, t); keep(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

In [ ]:
def accuracy(M, ctx):
    """RB-0's accuracy(): top-1 over the cached ILSVRC images, clean order, per-image correctness."""
    model, cache = M["model"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = model(images.to(DEVICE)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

In [ ]:
# setup
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        torch.manual_seed(0)   # identical to the seeded W8 test model, so test mode can be gated against a test RB-0 part F
        model = timm.create_model(spec["name"].split(".")[0], embed_dim=96, num_heads=6, num_classes=10, pretrained=False,
                                  dynamic_img_size=True).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0].to(DEVICE))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    rh, rh_src, nh_attr = attn_attrs(model, source)
    n_prefix = int(getattr(model, "num_prefix_tokens", 1))
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix, "n_heads": int(nh_attr), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg)
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    gate("model is the mixed-RoPE ViT (rope_mixed, learned freqs (2, 12, heads, 32))", key,
         bool(getattr(model, "rope_mixed", False)) and tuple(model.rope.freqs.shape[:3]) == (2, N_BLOCKS, M["n_heads"]) and (TEST_MODE or model.rope.freqs.shape[3] == 32),
         f"rope_mixed={getattr(model, 'rope_mixed', None)}, freqs {tuple(model.rope.freqs.shape)}, heads {M['n_heads']}")
    rope = model.rope; M["rope"] = rope; fr0 = rope.freqs.detach().clone()
    # which component is the row coordinate (RB-0 F's test, recomputed)
    D2 = None
    def emb(masks):
        with MixedSpec(rope, masks): return rope.get_embed(shape=(Hg, Wg)).detach().float()
    e_full = emb([]); D2 = e_full.shape[-1] // 2
    def variation(e):
        s = e[..., :D2].reshape(e.shape[0], e.shape[1], Hg, Wg, -1)
        return (s - s.mean(dim=2, keepdim=True)).abs().mean().item(), (s - s.mean(dim=3, keepdim=True)).abs().mean().item()
    v_rows0, v_cols0 = variation(emb([([1], None, None)]))           # only component 0 left
    comp0_is_col = v_rows0 < 1e-6 * max(v_cols0, 1e-12)
    ROWC, COLC = (1, 0) if comp0_is_col else (0, 1)
    M["ROWC"], M["COLC"] = ROWC, COLC
    gate("component 0 depends on exactly one grid axis", key, comp0_is_col or (v_cols0 < 1e-6 * max(v_rows0, 1e-12)),
         f"variation across rows {v_rows0:.2e}, across columns {v_cols0:.2e}")
    if RB0F is not None and not TEST_MODE:
        gate("row component equals RB-0 part F's", key, ROWC == RB0F["row_component"], f"freqs[{ROWC}] vs freqs[{RB0F['row_component']}]")
    e_none = emb([([0, 1], None, None)])
    gate("all frequencies zero gives exact identity rotations", key, torch.equal(e_none, identity_like(e_none)))
    e1 = emb([([0, 1], [8], [3])])
    others = torch.ones(N_BLOCKS, M["n_heads"], dtype=torch.bool); others[8, 3] = False
    gate("masking (block 8, head 3) makes exactly that head's rotations identity and leaves every other (block, head) bit-identical", key,
         torch.equal(e1[8, 3], identity_like(e1[8, 3])) and torch.equal(e1[others], e_full[others]))
    gate("frequency edits restore the exact tensor", key, torch.equal(rope.freqs.detach(), fr0))
    fx, fy = fr0[ROWC].abs().sum(-1), fr0[COLC].abs().sum(-1)                  # (depth, heads)
    M["orientation"] = ((fx - fy) / (fx + fy).clamp_min(1e-12)).tolist()
    M["orientation_l2"] = (((fr0[ROWC] ** 2).sum(-1) - (fr0[COLC] ** 2).sum(-1)) / ((fr0[ROWC] ** 2).sum(-1) + (fr0[COLC] ** 2).sum(-1)).clamp_min(1e-12)).tolist()
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); M["first_layer_rel_diff"] = v
        lo, hi = FIRST_LAYER_TF32_RANGE
        gate("first-layer numerics are TF32-level (cuDNN 9.10.2's kernel, as for the other five models; not exact fp32)", key,
             nb == BATCH_SIZE and lo <= v <= hi, f"{v:.4e} (batch {nb})")
    else:
        gate("first-layer numerics are TF32-level", key, True, "test mode / CPU", skip=True)
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("Cref", lambda: MixedSpec(rope, [window_mask("cref", ROWC, COLC)]))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    n_chk = min(512, N_ACC)
    dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk, batch_size=BATCH_SIZE, half=False, num_workers=0)
    cs = []
    with torch.inference_mode():
        for im, lb in dl: cs.append(model(im.to(DEVICE)).argmax(-1).cpu() == torch.as_tensor(lb))
    cs = torch.cat(cs).numpy().astype(np.uint8)
    _, cc, _ = accuracy(M, NullCtx())
    gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key, np.array_equal(cs, cc[:n_chk]),
         f"{int((cs != cc[:n_chk]).sum())} images differ")
    return M

In [ ]:
# bookkeeping
RESULTS = {}
CFG_FP = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET, "acc_dataset": ACC_DATASET,
          "metric": METRIC, "window": WINDOW, "post": POST, "p_ref": P_REF, "p_single": P_SINGLE, "noise_at": list(NOISE_AT),
          "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC], "noise": [NOISE_SEEDS, NOISE_BOOT], "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def model_file(key): return OUT_DIR / f"rb4_{key}.json"
def img_file(key): return OUT_DIR / f"rb4_perimage_{key}.npz"
def single_name(w, k, h): return f"{w}_b{k}_h{h}"
def single_readouts(k): return P_SINGLE if k < N_BLOCKS - 1 else [FINAL]

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB4_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    for n in R["acc"]:
        if n not in P: raise SystemExit(f"{img_file(key).name} lacks per-image results for {n}; set RB4_RESUME=0.")
    return R, P

def save_R(key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        tmpi = img_file(key).with_name(img_file(key).stem + ".tmp.npz"); np.savez_compressed(tmpi, **P); os.replace(tmpi, img_file(key))
    tmp = model_file(key).with_suffix(".json.tmp"); tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def windows_done(R, nh):
    return [w for w in WINDOW_ORDER if all(single_name(w, k, h) in R["conditions"] for k in POST for h in range(nh))]
def acc_plan(nh):
    arms = ["baseline", "floor", "Cref", "win_row", "win_col"]
    arms += [f"s_b{k}_h{h}" for k in POST for h in range(nh)] + [f"int_s_b{k}_h{h}" for k in POST for h in range(nh)]
    return arms

## Gates on the saved results

In [ ]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)

def check_refs(key, R, P):
    c = R["conditions"]; ok = []
    exp_hooks = {"baseline": 0, "Cref": 1, "Dref_row": 1, "Dref_col": 1}
    ok.append(gate("references registered the expected edits and saw every image", key,
                   all(c[n]["hooks"] == exp_hooks[n] and c[n]["n_images"] == N_SSDC for n in exp_hooks)))
    if RB0F is not None:
        f = RB0F["conditions"]; d, nv = 0.0, 0
        for n, src in RB0F_NAME.items():
            d = max(d, _maxdiff(c[n]["ssdc"], f[src]["ssdc"]))
            for b in ("11", "12"):
                for ax in ("row", "col"):
                    d = max(d, abs(c[n]["probes"][b][ax] - f[src]["probes"][b][ax])); nv += 1
        ok.append(gate("reference SSDC (13 readouts) and probes (entering 11, output of 11) equal RB-0 part F bit for bit", key,
                       d <= EXACT_TOL, f"max |delta| {d:.1e} over 4 x 13 SSDC + {nv} probes"))
        fa = RB0F.get("acc", {})
        accn = {"baseline": "baseline", "floor": "floor", "Cref": "Cref", "win_row": "win_row", "win_col": "win_col"}
        if all(n in R["acc"] for n in accn):
            dd = max(abs(R["acc"][n]["acc"] - fa[s]) for n, s in accn.items())
            ok.append(gate("top-1 references (baseline, floor, Cref, win_row, win_col) equal RB-0 part F bit for bit", key, dd == 0.0,
                           ", ".join(f"{n} {R['acc'][n]['acc'] * 100:.2f}" for n in accn)))
    nz = [c[n]["probes"][str(b)].get("noise", {}).get(ax, {}).get("combined_sd") for n in ("Cref", "Dref_row", "Dref_col") for b in NOISE_AT for ax in ("row", "col")]
    ok.append(gate("probe noise measured on the three window references (new) and finite", key, all(v is not None and np.isfinite(v) for v in nz)))
    return all(ok)

def check_singles(key, R, P):
    c = R["conditions"]; nh = R["n_heads"]; done = windows_done(R, nh)
    gate("windows with every single head measured", key, done == WINDOW_ORDER, f"done {done}")
    names = [(w, k, h) for w in done for k in POST for h in range(nh)]
    if names:
        bad_h = [single_name(*t) for t in names if c[single_name(*t)]["hooks"] != 2 or c[single_name(*t)]["n_images"] != N_SSDC]
        gate("every single-head condition registered window + head edits and saw every image", key, not bad_h, ",".join(bad_h[:5]))
        worst = max(_maxdiff(c[single_name(w, k, h)]["ssdc"][:k + 1], c[REFN[w]]["ssdc"][:k + 1]) for w, k, h in names)
        gate("single-head edits leave every readout at or before their block bit-identical to the window reference", key,
             worst <= EXACT_TOL, f"max |delta| {worst:.1e} over {len(names)}")
        inert = [single_name(w, k, h) for w, k, h in names if c[single_name(w, k, h)]["ssdc"][FINAL] == c[REFN[w]]["ssdc"][FINAL]]
        gate("every single-head edit changes the SSDC at the output of block 11 (no inert edit)", key, not inert, ",".join(inert[:5]))
        rd_bad = [single_name(w, k, h) for w, k, h in names if sorted(c[single_name(w, k, h)]["probes"]) != sorted(str(r) for r in single_readouts(k))]
        gate("single heads probed at the planned readouts", key, not rd_bad, ",".join(rd_bad[:5]))
    plan_ = acc_plan(nh); miss = [n for n in plan_ if n not in R["acc"]]
    gate("every planned top-1 arm measured", key, not miss, f"{len(plan_) - len(miss)}/{len(plan_)}")
    exp = lambda n: 0 if n == "baseline" else (1 if n in ("floor", "Cref", "win_row", "win_col") or n.startswith("int_s") else 2)
    badh = [n for n in R["acc"] if R["acc"][n]["hooks"] != exp(n) or R["acc"][n]["n_images"] != N_ACC]
    gate("every top-1 arm registered the expected edits and saw every image", key, not badh, ",".join(badh[:5]))
    if "cref" in done and all(f"s_b{k}_h{h}" in R["acc"] for k in POST for h in range(nh)):
        gate("frequency tensor unchanged at the end (baseline SSDC re-measured equals the first)", key,
             R.get("baseline_again") is not None and R["baseline_again"] == c["baseline"]["ssdc"], "")
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(s.get("cudnn") == EXPECT_CUDNN and s.get("torch") == EXPECT_TORCH for s in ses), f"{len(ses)} session(s)")

def attention_level_gate(M, window_name):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    kb, hb = POST[0], 0
    grab = {}
    proj = get_block_attention(get_vit_blocks(model, source)[kb], source).proj
    hk = proj.register_forward_pre_hook(lambda m, args: grab.__setitem__("x", args[0].detach().clone()))
    xb = M["cache"]["ssdc"][0][0][:4].to(DEVICE); wm = window_mask(window_name, M["ROWC"], M["COLC"])
    try:
        with torch.no_grad():
            with MixedSpec(M["rope"], [wm]): model(xb)
            ref_x = grab["x"]
            with MixedSpec(M["rope"], [wm, ([0, 1], [kb], [hb])]): model(xb)
            new_x = grab["x"]
    finally:
        hk.remove()
    hd = ref_x.shape[-1] // nh
    same = all(torch.equal(ref_x[..., h * hd:(h + 1) * hd], new_x[..., h * hd:(h + 1) * hd]) for h in range(nh) if h != hb)
    changed = not torch.equal(ref_x[..., hb * hd:(hb + 1) * hd], new_x[..., hb * hd:(hb + 1) * hd])
    gate(f"[{window_name} window] editing (block {kb}, head {hb}) changes that head's attention output and leaves the other {nh - 1} bit-identical",
         key, same and changed)

## Measurement
References (probes at blocks 7-11 and the output of block 11, probe noise on the three windows) and the top-1
references, then their gates (a failure stops the run when `RB4_STRICT=1`), then the 216 single-head probe conditions
and the 144 single-head top-1 arms. Saves after every block; a re-run continues where it stopped.

In [ ]:
def run_model(M):
    key, model, nh = M["key"], M["model"], M["n_heads"]; H, W = M["grid"]; rope = M["rope"]; ROWC, COLC = M["ROWC"], M["COLC"]
    R, P = load_R(key)
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "notebook": "RB-4", "config": CFG_FP, "n_heads": nh,
                "row_component": ROWC, "col_component": COLC, "orientation": M["orientation"], "orientation_l2": M["orientation_l2"],
                "conditions": {}, "acc": {}, "sessions": []}, {}
    else:
        print(f"  resuming: {len(R['conditions'])} probe conditions, {len(R['acc'])} top-1 arms saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
    t0 = time.time(); n_new = [0]
    def run(name, masks, readouts, noise_at=(), extra=None):
        if name in R["conditions"]: return R["conditions"][name]
        r = measure(M, MixedSpec(rope, masks) if masks else NullCtx(), capture=readouts)
        probes = {str(b): probe_block(r["acts"][b], H, W, b, heads=("row", "col"), noise=(b in noise_at)) for b in readouts}
        del r["acts"]
        rec = {"ssdc": r["ssdc"], "probes": probes, "hooks": r["hooks"], "n_images": r["n_images"], "session": RUN_STAMP}
        if extra: rec.update(extra)
        R["conditions"][name] = rec; n_new[0] += 1
        return rec
    def run_acc(name, masks):
        if name in R["acc"]: return
        a, c, hooks = accuracy(M, MixedSpec(rope, masks) if masks else NullCtx())
        R["acc"][name] = {"acc": a, "hooks": hooks, "n_images": int(len(c))}; P[name] = c; n_new[0] += 1
    print(f"\n--- references ---", flush=True)
    for name, w in [("baseline", None), ("Cref", "cref"), ("Dref_row", "row"), ("Dref_col", "col")]:
        rec = run(name, [window_mask(w, ROWC, COLC)] if w else [], P_REF, noise_at=(NOISE_AT if w else ()))
        p = rec["probes"]
        print(f"    {name:9s} b11 row {p['11']['row'] * 100:5.1f} col {p['11']['col'] * 100:5.1f} | out11 row {p['12']['row'] * 100:5.1f} "
              f"col {p['12']['col'] * 100:5.1f} | SSDC b11 {rec['ssdc'][11]:.6f}  {time.time() - t0:.0f}s", flush=True)
    for name, masks in [("baseline", []), ("floor", [([0, 1], None, None)]), ("Cref", [window_mask("cref", ROWC, COLC)]),
                        ("win_row", [window_mask("row", ROWC, COLC)]), ("win_col", [window_mask("col", ROWC, COLC)])]:
        run_acc(name, masks)
    print("    top-1:", {n: round(R["acc"][n]["acc"] * 100, 2) for n in ("baseline", "floor", "Cref", "win_row", "win_col")})
    save_R(key, R, P)
    refs_ok = check_refs(key, R, P)
    for w in ("cref", "row"): attention_level_gate(M, w)
    if not refs_ok and STRICT and not TEST_MODE:
        gate("single heads measured", key, False, "STOPPED: a reference gate failed (RB4_STRICT=1). Send the gate report.")
        return R
    print(f"\n--- single heads: probes under 3 windows, then top-1 ---", flush=True)
    t1 = time.time()
    for w in WINDOW_ORDER:
        ref = R["conditions"][REFN[w]]
        for k in POST:
            if all(single_name(w, k, h) in R["conditions"] for h in range(nh)): continue
            for h in range(nh):
                run(single_name(w, k, h), [window_mask(w, ROWC, COLC), ([0, 1], [k], [h])], single_readouts(k),
                    extra={"window": w, "block": k, "head": h})
            save_R(key, R, P)
            r_ = str(single_readouts(k)[0])
            costs = [((ref["probes"][r_]["row"] - R["conditions"][single_name(w, k, h)]["probes"][r_]["row"]) * 100,
                      (ref["probes"][r_]["col"] - R["conditions"][single_name(w, k, h)]["probes"][r_]["col"]) * 100, h) for h in range(nh)]
            br, bc = max(costs), max(costs, key=lambda t: t[1])
            print(f"    [{w}] block {k:2d} | largest row cost h{br[2]} {br[0]:5.2f}, col cost h{bc[2]} {bc[1]:5.2f} | {(time.time() - t1) / 60:.1f} min", flush=True)
    for k in POST:
        for h in range(nh):
            run_acc(f"s_b{k}_h{h}", [window_mask("cref", ROWC, COLC), ([0, 1], [k], [h])])
            run_acc(f"int_s_b{k}_h{h}", [([0, 1], [k], [h])])
        save_R(key, R, P)
        print(f"    top-1 block {k} done | {(time.time() - t1) / 60:.1f} min", flush=True)
    if "baseline_again" not in R:
        R["baseline_again"] = measure(M, NullCtx())["ssdc"]
    R["complete"] = windows_done(R, nh) == WINDOW_ORDER and all(n in R["acc"] for n in acc_plan(nh))
    save_R(key, R, P)
    check_singles(key, R, P)
    gate("model finished", key, True, f"{n_new[0]} new measurements, {(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    return R

In [ ]:
T_RUN = time.time()
for key in MODELS_RUN:
    R0, P0 = load_R(key)
    if R0 and R0.get("complete"):
        print(f"\n{key}: already complete; loading and re-checking"); RESULTS[key] = R0
        check_refs(key, R0, P0); check_singles(key, R0, P0); continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed: " + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")

## Analysis
Probe costs: points lost against the window's reference (RB-2's definitions). Top-1 costs: points against the 0-5
window (window arms) or the unmodified model (intact arms), paired per-image SEs.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def perm_spearman(x, y, n_perm=PERM_N, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rx, ry = rankdata(x), rankdata(y)
    if np.std(rx) == 0 or np.std(ry) == 0: return float("nan"), float("nan")
    rho = float(np.corrcoef(rx, ry)[0, 1]); g = np.random.default_rng(seed)
    null = np.array([np.corrcoef(rx, g.permutation(ry))[0, 1] for _ in range(n_perm)])
    return rho, float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))
def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
def label(rc, cc, bar):
    if rc > bar and rc - cc > bar: return "row"
    if cc > bar and cc - rc > bar: return "col"
    if rc > bar or cc > bar: return "mixed"
    return "none"
def tag(r): return "b11" if r == 11 else "out11"
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))

ANALYSIS, TABLES = {}, {"axis_map": [], "noise": []}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R or not all(n in R["conditions"] for n in REFN.values()): print(f"{key}: no references saved"); continue
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    c, nh = R["conditions"], R["n_heads"]; done = windows_done(R, nh); out = {"windows_done": done}
    def pr(name, r, ax): return c[name]["probes"][str(r)][ax]
    def sd(name, r, ax): return c[name]["probes"][str(r)].get("noise", {}).get(ax, {}).get("combined_sd")
    bar = {}
    for r in NOISE_AT:
        bar[("axis", r)] = 300 * max(sd("Dref_row", r, "row"), sd("Dref_col", r, "col"))
        bar[("cref", r)] = 300 * max(sd("Cref", r, "row"), sd("Cref", r, "col"))
        for n in REFN.values():
            for ax in ("row", "col"):
                TABLES["noise"].append({"model": key, "reference": n, "readout": tag(r), "probe": ax, "accuracy_pts": pr(n, r, ax) * 100,
                                        "combined_sd_pts": sd(n, r, ax) * 100, "bar_3sd_pts": 300 * sd(n, r, ax)})
    out["bars_pts"] = {f"{w}_{tag(r)}": v for (w, r), v in bar.items()}
    out["references_probe_pts"] = {n: {tag(r): {ax: pr(n, r, ax) * 100 for ax in ("row", "col")} for r in NOISE_AT} for n in ["baseline"] + list(REFN.values())}
    out["references_top1"] = {n: v["acc"] * 100 for n, v in R["acc"].items() if not n.startswith(("s_b", "int_s"))}
    orient = np.array(R["orientation"]); orient2 = np.array(R["orientation_l2"])
    rows = []
    for k in POST:
        for h in range(nh):
            row = {"model": key, "block": k, "head": h, "primary": tag(single_readouts(k)[0]), "orientation": float(orient[k, h]),
                   "orientation_l2": float(orient2[k, h])}
            for r in single_readouts(k):
                t_ = tag(r)
                if "cref" in done:
                    s = single_name("cref", k, h)
                    a, b = (pr("Cref", r, "row") - pr(s, r, "row")) * 100, (pr("Cref", r, "col") - pr(s, r, "col")) * 100
                    row.update({f"cref_row_cost_{t_}": a, f"cref_col_cost_{t_}": b, f"probe_score_{t_}": a + b})
                if "row" in done and "col" in done:
                    sr, sc = single_name("row", k, h), single_name("col", k, h)
                    rc = (pr("Dref_row", r, "row") - pr(sr, r, "row")) * 100; cc = (pr("Dref_col", r, "col") - pr(sc, r, "col")) * 100
                    row.update({f"row_cost_{t_}": rc, f"col_cost_{t_}": cc, f"favours_{t_}": label(rc, cc, bar[("axis", r)])})
            if "cref" in done:
                s = c[single_name("cref", k, h)]
                row["ssdc_cost_b11"] = c["Cref"]["ssdc"][11] - s["ssdc"][11]; row["ssdc_cost_out11"] = c["Cref"]["ssdc"][FINAL] - s["ssdc"][FINAL]
            if f"s_b{k}_h{h}" in P and "Cref" in P:
                row["top1_window_cost"], row["top1_window_se"] = paired(P["Cref"], P[f"s_b{k}_h{h}"])
            if f"int_s_b{k}_h{h}" in P and "baseline" in P:
                row["top1_intact_cost"], row["top1_intact_se"] = paired(P["baseline"], P[f"int_s_b{k}_h{h}"])
            p_ = row["primary"]
            for f_ in ("row_cost", "col_cost", "favours"):
                if f"{f_}_{p_}" in row: row[f_] = row[f"{f_}_{p_}"]
            rows.append(row)
    df = pd.DataFrame(rows); TABLES["axis_map"] += rows
    if "favours" in df:
        out["favours_counts"] = {grp: {l: int((sub.favours == l).sum()) for l in ("row", "col", "mixed", "none")}
                                 for grp, sub in (("blocks 6-10 (b11)", df[df.block < 11]), ("block 11 (out11)", df[df.block == 11]))}
        spec = df[df.favours.isin(["row", "col", "mixed"])].copy(); spec["max_cost"] = spec[["row_cost", "col_cost"]].max(axis=1)
        cols_ = ["block", "head", "primary", "row_cost", "col_cost", "favours", "orientation"] + [x for x in ("top1_window_cost", "top1_intact_cost") if x in spec]
        out["specialists"] = spec.sort_values("max_cost", ascending=False)[cols_].round(3).to_dict("records")
        d6 = df[df.block < 11]
        conc = {}
        for ax in ("row", "col"):
            pos = d6[f"{ax}_cost_b11"].clip(lower=0); tot = float(pos.sum())
            top = d6.loc[d6[f"{ax}_cost_b11"].idxmax()]
            conc[ax] = {"top_head": [int(top.block), int(top["head"])], "top_cost": float(top[f"{ax}_cost_b11"]),
                        "share_of_positive_total": float(top[f"{ax}_cost_b11"] / tot) if tot > 0 else float("nan"), "positive_total": tot}
        out["concentration_b11"] = conc
        rho, p = perm_spearman(d6["orientation"], d6["row_cost_b11"] - d6["col_cost_b11"])
        rho2, p2 = perm_spearman(d6["orientation_l2"], d6["row_cost_b11"] - d6["col_cost_b11"])
        sp6 = d6[d6.favours.isin(["row", "col"])]
        match = [(int(r_.block), int(r_["head"]), r_.favours, round(float(r_.orientation), 3),
                  bool((r_.favours == "row") == (r_.orientation > 0))) for _, r_ in sp6.iterrows()]
        out["orientation_test"] = {"rule": RULE_ORIENT, "spearman_L1": [rho, p], "spearman_L2": [rho2, p2],
                                   "specialists_axis_matches_orientation_sign": f"{sum(m[-1] for m in match)}/{len(match)}", "specialists": match,
                                   "orientation_all_heads_6_10": {"median_abs": float(np.median(np.abs(d6.orientation))),
                                                                   "frac_abs_above_0.5": float((np.abs(d6.orientation) > 0.5).mean())}}
    if "top1_window_cost" in df and "ssdc_cost_b11" in df:
        d6 = df[df.block < 11]; res = {"n_heads": int(len(d6))}
        res["ssdc_b11_vs_top1"] = perm_spearman(d6["ssdc_cost_b11"], d6["top1_window_cost"])
        if "probe_score_b11" in d6: res["probe_b11_vs_top1"] = perm_spearman(d6["probe_score_b11"], d6["top1_window_cost"])
        rk = d6.sort_values("ssdc_cost_b11", ascending=False).reset_index(drop=True)
        top = rk.iloc[0]; byw = df.sort_values("top1_window_cost", ascending=False).reset_index(drop=True)
        rank_top = int(byw.index[(byw.block == top.block) & (byw["head"] == top["head"])][0]) + 1
        cst = byw.iloc[0]
        res["rule_top"] = {"head": [int(top.block), int(top["head"])], "ssdc_cost_b11": float(top.ssdc_cost_b11),
                           "top1_window_cost": [float(top.top1_window_cost), float(top.top1_window_se)],
                           "top1_intact_cost": [float(top.get("top1_intact_cost", np.nan)), float(top.get("top1_intact_se", np.nan))],
                           "rank_in_window_top1": [rank_top, int(len(byw))], "favours_b11": top.get("favours_b11")}
        res["second_by_ssdc"] = [int(rk.iloc[1].block), int(rk.iloc[1]["head"]), float(rk.iloc[1].ssdc_cost_b11), float(rk.iloc[1].top1_window_cost)]
        res["costliest_window"] = [int(cst.block), int(cst["head"]), float(cst.top1_window_cost), float(cst.top1_window_se), cst.get("favours_b11"),
                                   float(cst.get("top1_intact_cost", np.nan))]
        nxt = byw.iloc[1]; a_, b_ = f"s_b{int(cst.block)}_h{int(cst['head'])}", f"s_b{int(nxt.block)}_h{int(nxt['head'])}"
        res["costliest_minus_next"] = [int(nxt.block), int(nxt["head"]), *paired(P[b_], P[a_])]
        if "top1_intact_cost" in df:
            mx = df.loc[df.top1_intact_cost.idxmax()]
            res["intact"] = {"max_any_head": [int(mx.block), int(mx["head"]), float(mx.top1_intact_cost), float(mx.top1_intact_se)],
                             "n_heads_above_3se": int(((df.top1_intact_cost - 3 * df.top1_intact_se) > 0).sum())}
            k_, h_ = int(cst.block), int(cst["head"])
            d = (P["Cref"].astype(float) - P[f"s_b{k_}_h{h_}"]) - (P["baseline"].astype(float) - P[f"int_s_b{k_}_h{h_}"])
            res["costliest_window_minus_intact"] = [float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)]
        out["top1"] = res
    ANALYSIS[key] = out

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}  ({MODEL_SPECS[key]['label']}) windows done: {out['windows_done']}\n" + "=" * 78)
    show("references: probes (pts) and top-1 (%)", {"probes": out["references_probe_pts"], "top1": out["references_top1"]})
    print("noise bars (3 SD, pts):", {k: round(v, 2) for k, v in out["bars_pts"].items()})
    if "favours_counts" in out:
        print("single heads by favoured axis:", out["favours_counts"])
        show("specialists, largest cost first", pd.DataFrame(out["specialists"]).head(20))
        show("concentration entering block 11 (the top head's share of all positive single-head cost)", out["concentration_b11"])
        show("is a head's rebuilt axis its learned frequency orientation?", {k: v for k, v in out["orientation_test"].items() if k != "rule"})
    if "top1" in out: show("top-1: the rule's top head, the costliest head, intact model", out["top1"])

In [ ]:
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    d = pd.DataFrame(TABLES["axis_map"])
    if len(d) and "favours" in d:
        fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
        colours = {"row": "#2f6db5", "col": "#d9822b", "mixed": "#7b4fa0", "none": "#a8a8a8"}
        b = ANALYSIS["mixed"]["bars_pts"]["axis_b11"]
        axs[0].add_patch(plt.Rectangle((-b, -b), 2 * b, 2 * b, color="#e6e6e6", zorder=0))
        for lab_, sub in d.groupby("favours"):
            axs[0].scatter(sub.row_cost, sub.col_cost, s=np.where(sub.block == 11, 18, 30), c=colours[lab_], label=lab_, edgecolors="none", zorder=2)
            axs[1].scatter(sub.orientation, sub.row_cost - sub.col_cost, s=np.where(sub.block == 11, 18, 30), c=colours[lab_], label=lab_, edgecolors="none")
        axs[0].set_xlabel("row cost (pts)"); axs[0].set_ylabel("column cost (pts)"); axs[0].set_title("mixed RoPE: every single head, blocks 6-11", fontsize=9)
        axs[1].set_xlabel("frequency orientation (+1 row only, -1 column only)"); axs[1].set_ylabel("row cost - column cost (pts)")
        axs[1].set_title("rebuilt axis vs learned frequency orientation", fontsize=9)
        for a_ in axs: a_.axhline(0, c="#888", lw=0.5); a_.axvline(0, c="#888", lw=0.5); a_.legend(fontsize=7, frameon=False)
        fig.tight_layout(); fig.savefig(OUT_DIR / "rb4_mixed_axis_map.png", dpi=150); plt.close(fig); print("figure written")
except Exception as ex:
    print("figure skipped:", repr(ex))

## Gate report and export

In [ ]:
_all = []
for f in sorted(OUT_DIR.glob("rb4_gates_session_*.json")): _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = bool((RESULTS.get("mixed") or {}).get("complete"))
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped | complete: {complete}")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows(): print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)
export = {"notebook": "RB-4", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo), "repo_commit": REPO_COMMIT,
          "rb0_dir": str(_rb0) if _rb0 else None, "config": {**CFG_FP, "strict": STRICT},
          "rules": {"bar": RULE_BAR, "label": RULE_LABEL, "rank": RULE_RANK, "orientation": RULE_ORIENT},
          "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
          "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete, "sessions": sorted(set(g["session"] for g in _all))}}
(OUT_DIR / f"rb4_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb4_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb4_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")